# Sample one patch per LC25000 class

For the CLIP contrastive-matrix diagram on slide 8. Grabs a random patch from each of the 5 LC25000 classes (colon_aca, colon_n, lung_aca, lung_n, lung_scc), triggers browser downloads.

Total run time: ~2 min including kagglehub download.

**After download**: files land in your Mac's Downloads folder as `lc25000_{class}.jpeg`. Reply to Claude with the paths and it'll swap them into the diagram.

In [ ]:
!pip install -q kagglehub

In [ ]:
from pathlib import Path
import shutil, glob, random

from google.colab import files
import kagglehub

random.seed(42)

OUT = Path('/content/lc_samples')
OUT.mkdir(exist_ok=True)
print(f'Outputs → {OUT}')

# Download LC25000 (~1.5 GB, but kagglehub caches so re-runs are instant)
lc_root = Path(kagglehub.dataset_download(
    'andrewmvd/lung-and-colon-cancer-histopathological-images'))
print(f'LC25000 at {lc_root}')

# 5 classes, glob patterns to match either flat or nested layout
CLASSES = {
    'colon_aca':  ['**/colon_image_sets/colon_aca/*.jpeg', '**/colon_aca/*.jpeg'],
    'colon_n':    ['**/colon_image_sets/colon_n/*.jpeg',   '**/colon_n/*.jpeg'],
    'lung_aca':   ['**/lung_image_sets/lung_aca/*.jpeg',   '**/lung_aca/*.jpeg'],
    'lung_n':     ['**/lung_image_sets/lung_n/*.jpeg',     '**/lung_n/*.jpeg'],
    'lung_scc':   ['**/lung_image_sets/lung_scc/*.jpeg',   '**/lung_scc/*.jpeg'],
}

saved = []
for cls, patterns in CLASSES.items():
    hits = []
    for pat in patterns:
        hits = sorted(glob.glob(f'{lc_root}/{pat}', recursive=True))
        if hits: break
    if not hits:
        print(f'  ⚠ {cls}: NOT FOUND')
        continue
    src = Path(random.choice(hits))
    dst = OUT / f'lc25000_{cls}.jpeg'
    shutil.copy(str(src), str(dst))
    print(f'  ✓ {cls}: {src.name} → {dst.name}')
    saved.append(dst)

print(f'\nSaved {len(saved)} patches to {OUT}')

In [ ]:
# Visual sanity check
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, len(saved), figsize=(3 * len(saved), 3))
if len(saved) == 1:
    axes = [axes]
for ax, p in zip(axes, saved):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem.replace('lc25000_', ''), fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Trigger browser downloads (one prompt per file)
for p in saved:
    files.download(str(p))